# Group 5 Challenge: Mini Data Preprocessing Pipeline

Your team is building a modular numeric data-cleaning pipeline for analytics workloads.
Fix the broken starter code across all three tasks so every assertion passes.

**Scoring:** Task 1 (`30 pts`) + Task 2 (`35 pts`) + Task 3 (`35 pts`) = **`100 pts`**.

In [ ]:
SCORES = {"task_1": 0, "task_2": 0, "task_3": 0}
print("Score tracker initialized. Complete Tasks 1 to 3 below!")

---
## Task 1: Encapsulation `[30 pts]` — Protect `DatasetBatch` State

Fix the `DatasetBatch` class below to enforce these rules:
1. `__init__(self, batch_name: str, scale_factor: float = 1.0)`:
   - Raise `ValueError("scale_factor must be strictly positive")` if `scale_factor <= 0`.
   - Store `self._batch_name`, `self._scale_factor = float(scale_factor)`, and `self._records = []`.
2. Read-only properties:
   - `batch_name` -> returns `self._batch_name`
   - `records` -> returns `tuple(self._records)` (prevents caller mutation!)
3. Validated `scale_factor` property:
   - `@property` `scale_factor` returns `self._scale_factor`.
   - `@scale_factor.setter` raises `ValueError("scale_factor must be strictly positive")` if `value <= 0`; else sets `self._scale_factor = float(value)`.
4. Method `add_record(self, value: float)`:
   - Raise `ValueError("record cannot be None")` if `value is None`.
   - Append `float(value)` to `self._records`.

In [ ]:
class DatasetBatch:
    def __init__(self, batch_name: str, scale_factor: float = 1.0):
        self.batch_name = batch_name
        self.scale_factor = float(scale_factor)
        self.records = []

    def add_record(self, value: float):
        self.records.append(float(value))

In [ ]:
# RUN THIS CELL TO VERIFY TASK 1 (30 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    bad_init = False
    try:
        DatasetBatch("B0", 0)
    except ValueError:
        bad_init = True
    assert bad_init, "DatasetBatch('B0', 0) must raise ValueError."

    db = DatasetBatch("Train-01", 2.0)
    assert hasattr(db, "_batch_name") and hasattr(db, "_scale_factor") and hasattr(db, "_records"), "Use protected attributes."
    assert db.batch_name == "Train-01" and db.scale_factor == 2.0, "Properties batch_name and scale_factor must return initial values."

    ro = False
    try:
        db.batch_name = "Other"
    except AttributeError:
        ro = True
    assert ro, "Property 'batch_name' must be read-only."

    assert isinstance(db.records, tuple), "Property 'records' must return a tuple."

    bad_set = False
    try:
        db.scale_factor = -1.5
    except ValueError:
        bad_set = True
    assert bad_set, "Setting scale_factor <= 0 must raise ValueError."

    bad_rec = False
    try:
        db.add_record(None)
    except ValueError:
        bad_rec = True
    assert bad_rec, "add_record(None) must raise ValueError."

    db.add_record(10)
    db.add_record(20)
    assert db.records == (10.0, 20.0), f"Expected (10.0, 20.0), got {db.records}."

    SCORES["task_1"] = 30
    print("[PASS] Task 1 (30 / 30 pts): DatasetBatch encapsulation verified.")
except Exception as e:
    SCORES["task_1"] = 0
    print(f"[FAIL] Task 1 (0 / 30 pts): {e}")

---
## Task 2: Inheritance `[35 pts]` — `ScaleStep` and `ClampStep`

Fix `ScaleStep` and `ClampStep` so they inherit from `BaseStep`:
1. `ScaleStep(BaseStep)`:
   - `__init__(self, step_name: str, factor: float)`: call `super().__init__(step_name)` and store `self.factor = factor`.
   - Override `apply(self, values: list) -> list`: return `[round(v * self.factor, 4) for v in values]`.
   - Extend `describe(self) -> str`: return `f"{super().describe()} (x{self.factor})"`.
2. `ClampStep(BaseStep)`:
   - `__init__(self, step_name: str, low: float, high: float)`: call `super().__init__(step_name)` and store `self.low = low`, `self.high = high`.
   - Override `apply(self, values: list) -> list`: return `[min(max(v, self.low), self.high) for v in values]`.
   - Extend `describe(self) -> str`: return `f"{super().describe()} ([{self.low}, {self.high}])"`.

In [ ]:
class BaseStep:
    def __init__(self, step_name: str):
        self._step_name = step_name

    def apply(self, values: list) -> list:
        raise NotImplementedError("Subclasses must implement apply()")

    def describe(self) -> str:
        return f"Step: {self._step_name}"


class ScaleStep(BaseStep):
    def __init__(self, step_name: str, factor: float):
        self.factor = factor

    def apply(self, values: list) -> list:
        return list(values)

    def describe(self) -> str:
        return f"Scale {self.factor}"


class ClampStep(BaseStep):
    def __init__(self, step_name: str, low: float, high: float):
        self.low = low
        self.high = high

    def apply(self, values: list) -> list:
        return list(values)

    def describe(self) -> str:
        return f"Clamp {self.low}..{self.high}"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 2 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert issubclass(ScaleStep, BaseStep) and issubclass(ClampStep, BaseStep), "Both steps must inherit from BaseStep."

    sc = ScaleStep("Normalize", 0.5)
    cl = ClampStep("Bound", 0.0, 10.0)

    assert sc.apply([4, 10, 20]) == [2.0, 5.0, 10.0], "ScaleStep.apply failed."
    assert cl.apply([-5, 5, 15]) == [0.0, 5, 10.0], "ClampStep.apply failed."
    assert sc.describe() == "Step: Normalize (x0.5)", f"Unexpected describe(): {sc.describe()}"
    assert cl.describe() == "Step: Bound ([0.0, 10.0])", f"Unexpected describe(): {cl.describe()}"

    SCORES["task_2"] = 35
    print("[PASS] Task 2 (35 / 35 pts): Inheritance and super() verified.")
except Exception as e:
    SCORES["task_2"] = 0
    print(f"[FAIL] Task 2 (0 / 35 pts): {e}")

---
## Task 3: Polymorphism and Duck Typing `[35 pts]` — `DropNegativesStep` & `run_pipeline`

Fix the code below to achieve clean polymorphism:
1. `DropNegativesStep` must **NOT** inherit from `BaseStep` (standalone duck-typed class).
   - Rename/implement `apply(self, values: list) -> list` returning `[v for v in values if v >= 0]`.
2. Refactor `run_pipeline(steps: list, values: list) -> list`:
   - Remove **all** `isinstance(...)` checks!
   - Sequentially apply `data = step.apply(data)` across `steps` without mutating the caller's original `values` list.

In [ ]:
class DropNegativesStep:
    def __init__(self, label: str = "drop_neg"):
        self.label = label

    def filter_values(self, values: list) -> list:
        return [v for v in values if v >= 0]


def run_pipeline(steps: list, values: list) -> list:
    data = list(values)
    for step in steps:
        if isinstance(step, ScaleStep):
            data = step.apply(data)
        elif isinstance(step, ClampStep):
            data = step.apply(data)
    return data

In [ ]:
# RUN THIS CELL TO VERIFY TASK 3 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert not issubclass(DropNegativesStep, BaseStep), "DropNegativesStep must NOT inherit from BaseStep."
    assert hasattr(DropNegativesStep, "apply"), "DropNegativesStep must implement apply()."

    assert "isinstance" not in run_pipeline.__code__.co_names, "Remove all isinstance() calls from run_pipeline()."

    raw = [-10, 4, 20, 50]
    steps = [
        DropNegativesStep(),          # [4, 20, 50]
        ScaleStep("Half", 0.5),       # [2.0, 10.0, 25.0]
        ClampStep("Cap", 0.0, 15.0),  # [2.0, 10.0, 15.0]
    ]
    out = run_pipeline(steps, raw)
    assert out == [2.0, 10.0, 15.0], f"Expected [2.0, 10.0, 15.0], got {out}."
    assert raw == [-10, 4, 20, 50], "Do not mutate original raw list."

    SCORES["task_3"] = 35
    print("[PASS] Task 3 (35 / 35 pts): Polymorphism and duck typing verified.")
except Exception as e:
    SCORES["task_3"] = 0
    print(f"[FAIL] Task 3 (0 / 35 pts): {e}")

---
## Final Scoreboard

In [ ]:
# RUN THIS CELL TO VIEW YOUR FINAL COMPETITION SCORE
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
total = SCORES["task_1"] + SCORES["task_2"] + SCORES["task_3"]
print("=" * 42)
print("DS212 GROUP COMPETITION SCOREBOARD")
print("=" * 42)
print(f"Task 1 (Encapsulation): {SCORES['task_1']:>2} / 30 pts")
print(f"Task 2 (Inheritance):   {SCORES['task_2']:>2} / 35 pts")
print(f"Task 3 (Polymorphism):  {SCORES['task_3']:>2} / 35 pts")
print("-" * 42)
print(f"TOTAL SCORE:           {total:>3} / 100 pts")
if total == 100:
    print("STATUS: ALL CHALLENGES COMPLETED [100%]")
else:
    print("STATUS: IN PROGRESS — KEEP DEBUGGING!")
print("=" * 42)